# LangChain을 활용한 AI 에이전트 개발 심화
## 1일차 — LangChain Agent 기초



## 에이전트 정의와 활용 사례

에이전트는 사용자의 요청을 보고 **다음 행동을 스스로 선택하는 LLM 기반 실행 구조**입니다.  
항상 바로 답하는 것이 아니라, 필요하면 검색, 계산, 파일 읽기, 외부 시스템 호출 같은 Tool을 사용한 뒤 그 결과를 바탕으로 답합니다.

```text
사용자 요청
-> LLM이 필요한 행동 판단
-> 직접 답변 또는 Tool 사용
-> Tool 결과를 반영해 최종 답변
```

일반 챗봇이 주로 답변 생성에 집중한다면, 에이전트는 **답변 생성 + 필요한 행동 선택**까지 포함합니다.

대표 활용 사례는 다음과 같습니다.

| 활용 사례 | 에이전트가 하는 일 |
|------|------|
| 정보 검색 | 웹, 문서, 데이터베이스에서 필요한 정보를 찾아 근거 기반 답변 생성 |
| 데이터 분석 | CSV나 업무 데이터를 집계, 비교, 요약해 분석 결과 작성 |
| 고객 문의 처리 | 주문 상태, 반품 규정, 고객 문의 파일을 확인하고 답변 작성 |
| 승인형 자동화 | 메일 발송, 환불 처리처럼 위험한 작업은 사람 승인 후 실행 |

모델 호출과 메시지 구조를 확인한 뒤, Tool, Structured Output, Memory를 붙여 기본 에이전트를 만들어 봅니다.




## 0. 환경 설정

### 0.1 OpenAI API 키와 `.env` 파일

앞에서 에이전트는 LLM을 기반으로 사용자의 요청을 이해하고 다음 행동을 선택한다고 했습니다.  
이번 실습에서는 그 LLM을 Python 코드에서 사용하기 위해 **OpenAI API 키**를 설정합니다.

API는 한 프로그램이 다른 프로그램의 기능을 **요청/응답 방식으로 사용하게 해주는 인터페이스**입니다.  
즉, 사용자가 입력한 질문을 Python 코드가 OpenAI 서버로 보내고, 서버가 생성한 답변을 다시 받아 화면에 보여주는 구조라고 이해하면 됩니다.  
LLM은 사용자의 요청을 이해하고 답변이나 다음 행동을 결정하는 에이전트의 엔진 역할을 합니다.  
큰 LLM을 내 컴퓨터에서 직접 실행하려면 고성능 GPU가 필요하므로, 실습에서는 OpenAI 서버의 모델을 API로 호출합니다.

핵심만 정리하면:
- API 요금은 **사용량 기반 과금**입니다.
- ChatGPT Plus/Pro 같은 웹 구독과 **API 요금은 별도**입니다.
- API 키는 요청 주체와 사용량을 식별하는 **인증 키**입니다.
- 실습 폴더 루트의 `.env`에는 `OPENAI_API_KEY=...` 한 줄이면 됩니다.

**대표 모델과 요금 감각**  
(2026년 4월 17일 기준, Standard text token price, 1M tokens 기준)

| 모델 | 입력 가격 | 출력 가격 | 추천 용도 |
|------|----------:|----------:|-----------|
| `gpt-4o-mini` | **$0.15** | **$0.60** | FAQ 챗봇, 가벼운 분류/추출 |
| `gpt-4.1-mini` | **$0.40** | **$1.60** | 지시 이행, Tool 호출, 실무형 API |
| `gpt-4.1-nano` | **$0.10** | **$0.40** | 초저비용 분류, 간단한 자동화 |
| `gpt-5-mini` | **$0.25** | **$2.00** | 범용 실습, 저지연 서비스 |
| `gpt-5-nano` | **$0.05** | **$0.40** | 입문 실습, 요약, 분류, 대량 처리 |
| `gpt-5.4-mini` | **$0.75** | **$4.50** | 최신 고성능 경량 워크로드 |
| `gpt-5.4-nano` | **$0.20** | **$1.25** | 최신 고볼륨 경량 워크로드 |

이번 노트북의 기본 실습 모델은 `gpt-5-nano`로 둡니다.  
비용 부담이 작고 예제가 빠르게 돌아가며, 1일차 실습 난도에는 충분합니다.

최신 모델 목록과 상세 요금은 아래 공식 문서에서 확인하세요.
- https://developers.openai.com/api/docs/models
- https://developers.openai.com/api/docs/pricing

**발급 절차**

1. https://platform.openai.com/ 접속  
2. 로그인 또는 계정 생성  
3. Billing에서 결제 수단 또는 prepaid credits 설정  
4. API key 페이지에서 `Create new secret key` 클릭  
5. 발급된 키를 바로 복사해 안전한 위치에 보관

발급된 키는 한 번만 표시되므로, 바로 `.env` 파일이나 안전한 비밀 저장소에 보관하는 편이 좋습니다.

실습 폴더 루트의 `.env` 파일에는 아래 한 줄만 넣으면 됩니다.

```bash
OPENAI_API_KEY=...
```

**관리 원칙**
- API 키를 노트북 셀에 직접 하드코딩하지 않습니다.
- Git 저장소나 메신저에 공유하지 않습니다.
- 발급된 API 키는 안전한 위치에 보관합니다.
- Billing과 Usage Limits에서 예산과 한도를 함께 확인합니다.


In [ ]:
# 1일차 공통 준비 셀입니다.
import os  # os.environ으로 환경 변수를 읽습니다.
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv  # .env 파일 값을 현재 세션으로 불러옵니다.
from langchain_openai import ChatOpenAI  # LangChain에서 쓰는 채팅 모델 래퍼입니다.

load_dotenv(override=True)  # 현재 폴더의 .env 값을 우선 적용해 환경 변수에 올립니다.
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요."  # 키가 없으면 여기서 바로 멈춥니다.

lf_config = {}  # 이후 callback 등을 붙일 때 함께 넘길 공통 config입니다.

## 1. API 호출과 프롬프트

먼저 OpenAI API로 모델을 한 번 호출해보고, 이어서 앱에서 쓰는 `System / User / Assistant` 메시지 구조를 확인합니다.


### 1.1 OpenAI API 직접 호출

가장 단순한 출발점은 모델에게 한 번 요청하고 한 번 응답을 받는 구조입니다.  
OpenAI SDK에서는 보통 아래처럼 이해하면 쉽습니다.
- `instructions` = system 역할에 가까운 지시
- `input` = user 역할에 가까운 현재 요청

다음 단계는 이 호출 앞뒤에 **프롬프트 구성과 결과 정리 단계**를 붙여 체인으로 만드는 것입니다.


In [ ]:
# LangChain 없이 OpenAI SDK만으로 호출하는 최소 예제입니다.
from openai import OpenAI  # OpenAI SDK 클라이언트를 예제 셀에서 직접 만듭니다.

client = OpenAI()
direct_answer = client.responses.create(
    model="gpt-4.1-nano",  # 사용할 모델 이름입니다.
    instructions="당신은 친절한 설명자입니다.",  # 시스템 역할에 해당합니다.
    input="LangChain이 무엇인지 한 문단으로 설명해줘.",  # 실제 사용자 요청입니다.
)

print("[직접 모델 호출]")
print(direct_answer.output_text)  # 응답 본문만 간단히 꺼내 봅니다.

OpenAI SDK만으로도 모델 호출은 충분히 할 수 있습니다.  
다만 실제 앱에서는 호출 한 번으로 끝나지 않는 경우가 많습니다.

모델을 부르기 전에 사용자 입력과 규칙을 프롬프트로 재구성하고, 응답을 받은 뒤 필요한 형태로 정리하고, 여러 단계를 이어 붙이고, 필요하면 Tool이나 Memory도 연결해야 합니다.  
LangChain은 이런 반복 구조를 매번 직접 구현하지 않도록 공통 인터페이스로 묶어줍니다.

그 출발점이 메시지 역할을 나누는 것입니다.  
앱은 보통 질문 문자열 하나만 보내지 않고, **항상 지킬 규칙**, **현재 사용자 요청**, **이전 대화 흐름**을 구분해서 모델에 전달합니다.



### 1.2 앱은 질문만 보내지 않습니다

일반 ChatGPT에서는 보통 대화창에 입력한 질문이나 요청 전체를 프롬프트라고 생각합니다.  
그 요청은 한 줄일 수도 있고 여러 줄일 수도 있습니다.

하지만 앱에서는 그 요청만 단독으로 보내지 않고, 한 번의 호출에도 **역할이 다른 메시지들**을 함께 묶어 모델에 전달합니다.

왜 나눠 보내느냐 하면, **규칙과 이번 요청과 이전 답변을 한 문장에 섞어 쓰면  
무엇이 지시이고 무엇이 현재 요청이고 무엇이 이전 맥락인지 흐려지기 쉽기** 때문입니다.

역할을 나눠 두면 모델도 **항상 지킬 규칙**, **지금 처리할 요청**, **직전 대화 흐름**을 더 분명하게 받아들일 수 있습니다.


### 1.3 System / User / Assistant

| 이름 | LangChain에서 자주 보이는 형태 | 역할 |
|------|--------------------------|------|
| `System` | `system` / `SystemMessage` | 모델의 역할과 규칙을 정합니다. |
| `User` | `user` / `HumanMessage` | 지금 사용자가 요청한 내용을 넣습니다. |
| `Assistant` | `assistant` / `AIMessage` | 직전 답변이나 few-shot 예시를 넣습니다. |

`System / User / Assistant`는 메시지 구조를 이해할 때 가장 기본이 되는 세 가지입니다.  
에이전트를 만들면 여기에 Tool 실행 결과 메시지도 더해집니다.


모델을 호출할 때는 시스템 프롬프트, 사용자 메시지, 이전 AI 메시지가 하나의 `messages` 묶음으로 전달됩니다.  
시스템 프롬프트는 모델이 먼저 지켜야 하는 규칙이고, 사용자 요청은 그 규칙 안에서 해석됩니다.

<img src="image/messages_to_model_context.svg" width="760">




아래 예시는 모델을 호출하지 않고, `ChatPromptTemplate.from_messages()`가 메시지를 어떻게 묶는지만 확인합니다.
`{question}`처럼 중괄호로 감싼 값은 실행할 때 실제 입력으로 채워집니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 쇼핑몰 고객센터 직원처럼 답한다."),
    ("user", "{question}"),
])

messages_preview = basic_prompt.invoke({"question": "환불은 언제까지 신청할 수 있어?"})


for message in messages_preview.messages:
    print(type(message).__name__, ":", message.content)


### 1.4 `user`와 `human`은 왜 둘 다 보일까?

OpenAI 계열 API 설명에서는 보통 `system / user / assistant`를 씁니다.  
LangChain 문서와 예제에서는 `HumanMessage`처럼 `human`이라는 이름도 자주 보입니다.

핵심은 둘 다 **사람 쪽 입력 메시지**를 가리킨다는 점입니다.  
혼란을 줄이기 위해 설명에서는 가능하면 `user`로 통일합니다.


## 2. LangChain 체인

앞에서 역할별 메시지를 나눠 보냈다면, 이제 그 메시지 구성 -> 모델 호출 -> 결과 정리 과정을 하나의 고정된 흐름으로 묶어봅니다.  
즉 체인은 여러 과정을 하나의 고정된 흐름으로 묶어, 필요할 때마다 같은 순서로 실행하는 객체입니다.


### 2.1 체인은 언제 쓰는가?

모델 단독 호출은 한 번 요청하고 한 번 응답을 받는 가장 단순한 방식입니다.  
체인은 그 앞뒤에 필요한 단계를 붙여 **정해진 흐름을 반복 실행**할 때 잘 맞습니다.

에이전트는 질문을 보고 다음에 무엇을 할지 고르는 더 유연한 구조입니다.  
예를 들어 `현재 시간을 알려줘`처럼 외부 정보가 필요한 요청은 시간 조회 함수를 사용하고, `이 문장을 요약해줘`처럼 모델 답변만으로 충분한 요청은 바로 답하는 식입니다.  
체인은 그와 달리 **정해진 단계를 코드로 묶어 반복 실행하는 방식**입니다.

<img src="image/체인과에이전트.png" width="700">


### 2.2 LCEL 체인

방금 본 system/user 메시지 구성을 그대로 재사용해, LangChain에서는 LCEL로 단계를 `|` 연산자로 연결합니다.  
이 예제는 `ChatPromptTemplate`으로 system/user 메시지를 묶고, 모델 호출 뒤에 `StrOutputParser`를 붙여 문자열만 꺼내는 흐름입니다.

```text
messages 입력 -> ChatPromptTemplate -> 모델 -> StrOutputParser
```

체인의 장점은 **중간 단계를 눈에 보이게 만들고 통제하기 쉽다**는 점입니다.


In [ ]:
# LCEL에서는 메시지 구성 -> 모델 호출 -> 결과 정리를 | 로 이어 하나의 체인으로 만듭니다.
from langchain_core.prompts import ChatPromptTemplate  # system/user 메시지를 한 번에 묶습니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답을 문자열로 정리합니다.

model = ChatOpenAI(model="gpt-4.1-nano")  # 가볍게 실습하기 좋은 OpenAI 채팅 모델입니다.
print("모델 준비 완료:", model.model_name)

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 기술 개념을 쉬운 한국어로 설명하는 도우미다."),
    ("user", "{topic}이 무엇인지 한 문단으로 설명해줘."),
])
parser = StrOutputParser()

chain = prompt | model | parser
chain_result = chain.invoke({"topic": "LangChain"}, config=lf_config)

print("[체인 호출]")
print(chain_result)

## 3. Prompt, Messages와 Context

체인은 정해진 단계를 따라가므로 프롬프트와 입력이 비교적 단순합니다.  
하지만 에이전트로 가면 현재 사용자 질문만이 아니라 시스템 메시지, 이전 대화, Tool 설명처럼 **모델에 무엇을 함께 전달할지**가 더 중요해집니다.  
즉, 모델이 어떤 형식으로 정보를 받고, 그중 무엇을 바탕으로 판단하는지부터 먼저 익혀두는 편이 좋습니다.

자주 헷갈리는 용어는 이렇게 구분하면 됩니다.

| 용어 | 한 줄 구분 | 예시 |
|------|------------|------|
| `Prompt` | 좁게는 모델에게 주는 지시문이나 요청 문장 | 시스템 프롬프트, 사용자 요청 |
| `Messages` | 역할과 내용을 함께 담는 대화 입력 형식 | `system`, `user`, `assistant` |
| `Context` | 모델이 이번 호출에서 참고하는 정보 전체 | messages, 이전 대화, 검색 문서, Tool 결과 |

프롬프트라는 말은 넓게는 모델 입력 전체를 뜻하기도 합니다.  
혼란을 줄이기 위해 여기서는 **지시문이나 요청 문장**이라는 좁은 의미로 사용합니다.


### 3.1 Context란?

모델이 이번 호출에서 참고하는 **정보 전체**를 `Context`라고 생각하면 됩니다.  
여기에는 프롬프트가 담긴 메시지 목록과 답변에 필요한 참고 정보가 함께 들어갈 수 있습니다.

예를 들어 쇼핑몰 반품 문의에 답할 때 한 번의 모델 호출에는 아래 정보가 함께 들어갈 수 있습니다.

```text
Context
├─ system: 너는 쇼핑몰 고객 문의에 짧고 정확하게 답하는 도우미다.
├─ user: 단순 변심으로 반품하면 배송비는 누가 부담해?
├─ 이전 대화: 사용자는 어제 받은 티셔츠 반품을 문의하고 있다.
└─ 참고 정보: 단순 변심 반품 배송비는 고객 부담이다.
```

`Messages`는 이 Context 안에서 `system`, `user`, `assistant`처럼 역할을 나눠 담는 형식입니다.  
모델은 이 정보들을 함께 보고 지금 어떤 답을 해야 할지 판단합니다.




### 3.2 Prompt, Messages와 Context의 관계

체인 예제에서는 `Prompt`라는 말을 자주 씁니다.  
`ChatPromptTemplate`처럼 모델에게 보낼 지시문과 사용자 입력을 구성하는 단계가 중심이기 때문입니다.

에이전트 예제에서는 `Context`라는 말을 더 자주 보게 됩니다.  
에이전트는 사용자 요청뿐 아니라 시스템 메시지, 이전 대화, Tool 설명, Tool 결과처럼 모델이 판단할 때 참고하는 정보를 함께 다루기 때문입니다.

정의는 다음처럼 잡으면 됩니다.

- `Prompt`: 모델에게 일을 시키는 지시문이나 요청 문장입니다. 체인에서는 입력 구성을 `prompt`라고 많이 부릅니다.
- `Messages`: `system`, `user`, `assistant`처럼 역할과 내용을 나눠 담는 대화 형식입니다.
- `Context`: 모델이 이번 호출에서 참고하는 정보 전체입니다. 에이전트에서는 이 표현을 더 자주 씁니다.

아래 그림은 세 용어의 포함 관계를 간단히 보여줍니다.

<img src="image/messages_context_venn.svg" width="560">

`Messages`는 모델에게 정보를 전달하는 대표적인 형식입니다.  
각 메시지는 `role`과 `content`로 나뉘고, `content` 안에 시스템 프롬프트나 사용자 요청이 들어갑니다.




### 3.3 Prompt Engineering vs Context Engineering

`Prompt Engineering`은 **문장을 어떻게 쓸지** 다듬는 일입니다.  
예를 들어 `짧게 답해줘`, `표로 정리해줘`, `근거를 먼저 말해줘`처럼 답변 방식과 규칙을 조정합니다.

`Context Engineering`은 **모델이 참고할 정보 전체를 어떻게 구성할지** 정하는 일입니다.  
어떤 메시지를 넣을지, 이전 대화를 어디까지 포함할지, 현재 작업에 필요한 참고 정보를 함께 줄지 결정합니다.

| 개념 | 초점 | 예시 |
|------|------|------|
| Prompt Engineering | 지시문과 표현 방식 | 시스템 프롬프트 문구 다듬기 |
| Context Engineering | 모델이 참고할 정보 구성 | Messages 구성, 이전 대화 범위, 참고 정보 선택 |

정리하면 프롬프트는 Context 안에 들어가는 지시문이고, Context는 그 지시문을 포함해 모델이 보는 전체 입력 범위입니다.




### 3.4 컨텍스트 설계에서 확인할 점

- 현재 질문에 필요한 히스토리만 포함했는지 확인합니다.
- 규칙이 많거나 서로 충돌할 수 있으면 중요한 기준이 드러나는지 확인합니다.
- 사용자별 대화가 서로 섞이지 않도록 스레드를 구분했는지 확인합니다.


### 3.5 컨텍스트 설계 예시

컨텍스트 관리는 정확도뿐 아니라 비용과 속도에도 영향을 줍니다.

문제가 되는 경우는 보통 두 가지입니다.

- 오래된 규정과 현재 규정이 함께 들어가 답변 기준이 흔들린다.
- 질문과 상관없는 안내문까지 함께 들어가 입력 토큰과 응답 시간이 늘어난다.

예를 들어 고객이 무료배송 주문을 일부 반품하려고 할 때,  
모델에게 필요한 정보는 **현재 부분 반품 규정**입니다.

하지만 실제 시스템에서는 예전 규정, 전체 반품 안내, 쿠폰 안내, 교환 안내처럼 비슷해 보이는 문서가 함께 검색될 수 있습니다.  
아래 코드는 **관리 안 된 Context**와 **정리된 Context**의 답변, 토큰 사용량, 실행 시간을 비교합니다.



In [ ]:
# Context가 충돌하거나 불필요하게 길어지는 상황을 비교합니다.
import time
from langchain_core.prompts import ChatPromptTemplate

question = "무료배송으로 6만원을 주문했는데, 2만원짜리 상품 하나만 반품하면 처음 배송비도 차감돼?"

context_prompt = ChatPromptTemplate.from_messages([
    ("system", "쇼핑몰 상담원처럼 답하세요. Context 안의 규정을 근거로 하되, 충돌하는 내용이 있으면 어느 규정을 기준으로 삼았는지도 짧게 말하세요."),
    ("user", "Context:\n{context}\n\n질문: {question}"),
])

context_chain = context_prompt | model

In [ ]:
# 관리 안 된 Context: 충돌하는 규정과 불필요한 안내가 함께 들어간 경우입니다.
messy_context = """
[예전 무료배송 반품 규정]
무료배송 주문에서 일부 상품을 반품해도 최초 배송비는 차감하지 않습니다.
단순 변심 반품 배송비만 고객이 부담합니다.

[현재 무료배송 주문 부분 반품 규정]
무료배송 주문에서 일부 상품을 반품해 최종 구매금액이 50,000원 미만이 되면,
최초 배송비 3,000원을 환불액에서 차감합니다.
단순 변심 반품 배송비는 고객이 부담합니다.

[교환 안내]
사이즈 교환은 상품 수령 후 7일 이내 신청할 수 있습니다.
착용 흔적이나 세탁 흔적이 있으면 교환이 제한됩니다.

[쿠폰 안내]
쿠폰은 주문당 1개만 사용할 수 있습니다.
이미 사용한 쿠폰은 주문 취소 후에도 재발급되지 않을 수 있습니다.
""".strip()

In [ ]:
# 관리 안 된 Context로 실행하고, 응답 metadata에서 토큰 사용량을 확인합니다.
started_at = time.perf_counter()
messy_response = context_chain.invoke({"context": messy_context, "question": question}, config=lf_config)
messy_elapsed = time.perf_counter() - started_at
messy_usage = messy_response.usage_metadata or {}

print("[관리 안 된 Context 답변]")
print(messy_response.content)
print("\n[실행 정보]")
print("입력 토큰:", messy_usage.get("input_tokens"))
print("출력 토큰:", messy_usage.get("output_tokens"))
print("전체 토큰:", messy_usage.get("total_tokens"))
print("실행 시간(초):", round(messy_elapsed, 2))

In [ ]:
# 정리된 Context: 현재 질문에 필요한 규정만 남긴 경우입니다.
clean_context = """
[현재 무료배송 주문 부분 반품 규정]
무료배송 주문에서 일부 상품을 반품해 최종 구매금액이 50,000원 미만이 되면,
최초 배송비 3,000원을 환불액에서 차감합니다.
단순 변심 반품 배송비는 고객이 부담합니다.
""".strip()

In [ ]:
# 정리된 Context로 실행하고, 응답 metadata에서 토큰 사용량을 확인합니다.
started_at = time.perf_counter()
clean_response = context_chain.invoke({"context": clean_context, "question": question}, config=lf_config)
clean_elapsed = time.perf_counter() - started_at
clean_usage = clean_response.usage_metadata or {}

print("[정리된 Context 답변]")
print(clean_response.content)
print("\n[실행 정보]")
print("입력 토큰:", clean_usage.get("input_tokens"))
print("출력 토큰:", clean_usage.get("output_tokens"))
print("전체 토큰:", clean_usage.get("total_tokens"))
print("실행 시간(초):", round(clean_elapsed, 2))

### 3.6 컨텍스트 제어 전략

1. 메시지 트리밍: 최근 메시지 중심으로 유지
2. 요약 기반 압축: 긴 대화를 요약문으로 치환
3. 역할별 프롬프트 분리: 관리자/일반 사용자 응답 전략 구분
4. Tool 제한: 현재 상황에 필요한 Tool만 노출
5. 상태 분리: 세션 상태와 장기 메모리를 분리 설계

핵심은 많이 넣는 것이 아니라, **지금 필요한 정보만 정확히 넣는 것**입니다.


## 4. Tool과 create_agent

모델이 어떤 메시지와 정보를 보게 할지 설계했다면, 다음은 바깥 정보와 기능을 연결할 차례입니다.  
최신 정보 조회, 계산, 실행처럼 모델만으로 처리하기 어려운 문제는 Tool이 필요합니다.

에이전트는 사용자의 요청을 보고 바로 답할 수 있는지, 바깥 정보나 기능이 필요한지 판단합니다.  
필요하면 Tool을 호출하고, Tool 결과를 바탕으로 최종 답변을 만듭니다.

이 절에서는 검색 활용 예를 먼저 보고, Tool을 어떻게 정의하는지 살펴본 뒤, 마지막에 `create_agent`로 그 Tool을 에이전트에 연결해봅니다.

<img src="image/일반LLM과도구에이전트차이.png" width="700">


### 4.1 Tool

Tool은 모델만으로 해결하기 어려운 문제를 처리하기 위해 바깥 정보나 기능을 연결하는 장치입니다.  
최신 정보 조회, 데이터베이스 검색, 계산, 외부 시스템 실행처럼 모델 밖의 도움이 필요할 때 사용합니다.

대표 예:
- 오늘 날씨, 현재 주가, 환율처럼 최신 사실이 필요한 경우
- 쇼핑몰 규정, 주문 DB, 고객 문의 CSV처럼 외부 데이터를 찾아야 하는 경우
- 계산, 예약, 티켓 생성처럼 실제 기능 실행이 필요한 경우

즉, Tool은 **최종 답변의 모양**을 정하는 기능이 아니라, **모델이 무엇을 할 수 있는지**를 넓혀 주는 기능입니다.


<img src="image/chatgpt_search.png" width="500">

<날씨 질문에 답변하기 위해 chatgpt가 검색을 활용하는 모습>

이처럼 모델이 바깥 기능을 써야 할 때 Tool이 필요합니다.  
이제 Tool을 어떻게 설계하고 정의하는지부터 봅니다.



### 4.2 Tool을 잘 정의하는 법

모델은 Tool 이름, 입력 형식, docstring을 보고 **언제 써야 하는지**와 **어떤 값을 넣어야 하는지**를 판단합니다.  
그래서 Tool 설명은 길기보다 분명해야 합니다.

- 이름: 기능이 바로 드러나게 적습니다. `lookup`보다 `lookup_order_status`가 낫습니다.
- 입력: 필요한 값을 필드로 나눕니다. `query: str` 하나보다 `order_id`, `date`처럼 나누는 편이 낫습니다.
- docstring: 언제 써야 하는지 짧고 분명하게 적습니다. "데이터를 조회합니다."보다 "주문번호로 배송 상태를 조회합니다."가 낫습니다.
- 반환값과 실패 메시지: 다음 단계가 판단하기 쉽게 애매하지 않게 돌려줍니다.

### 4.3 Tool 정의 문법

LangChain에서는 보통 일반 Python 함수에 `@tool`을 붙여 Tool을 정의합니다.  
함수 이름, 입력 타입, docstring이 합쳐져 Tool 인터페이스가 됩니다.

```python
from langchain.tools import tool

@tool
def lookup_order_status(order_id: str) -> str:
    """주문번호로 배송 상태를 확인해야 할 때 사용합니다."""
    return "배송 중입니다. 오늘 오후 도착 예정입니다."
```

**이 함수가 모델에게 그대로 전달되는 것은 아닙니다.**  
LangChain은 함수 정보를 읽어 LLM이 이해할 수 있는 Tool schema로 바꿔 모델 호출에 함께 보냅니다.

```json
{
  "type": "function",
  "function": {
    "name": "lookup_order_status",
    "description": "주문번호로 배송 상태를 확인해야 할 때 사용합니다.",
    "parameters": {
      "type": "object",
      "properties": {
        "order_id": {
          "type": "string"
        }
      },
      "required": ["order_id"]
    }
  }
}
```

모델은 이 정보를 보고 판단합니다.

- 함수 이름은 Tool 이름이 됩니다.
- 괄호 안의 입력값 이름과 타입이 Tool 입력 형식이 됩니다. 예를 들어 `order_id: str`은 `order_id`라는 문자열을 입력으로 받는다는 뜻입니다.
- docstring은 사용 기준 설명이 됩니다.
- 함수 내부 코드나 데이터 전체는 먼저 모델에게 보이지 않습니다.

예를 들어 사용자가 `주문번호 1001 배송 상태 알려줘`라고 물으면, 모델은 `description`과 `order_id` 입력 형식을 보고 `lookup_order_status`를 호출해야겠다고 판단합니다.  
Tool이 실행된 뒤 반환값이 다시 모델에게 전달되고, 모델은 그 결과를 바탕으로 최종 답변을 만듭니다.

입력이 복잡해지면 `args_schema`에 Pydantic 모델을 붙여 더 명확하게 정의할 수 있습니다.

### 4.4 Tool 실패 사례

반대로 Tool 인터페이스가 흐리면 에이전트도 쉽게 흔들립니다.

```python
from langchain.tools import tool

@tool
def lookup(query: str) -> str:
    """데이터를 조회합니다."""
    return "조회 결과"
```

이 Tool은 모델에게 필요한 힌트를 거의 주지 못합니다.  
`lookup`이라는 이름만으로는 주문 조회인지, 고객 검색인지, 문서 검색인지 알기 어렵고, `query` 하나만 있으면 어떤 값을 넣어야 하는지도 모호합니다.  
docstring도 "데이터를 조회합니다"라고만 되어 있어, 모델은 이 Tool을 언제 써야 하고 언제 쓰면 안 되는지 구분하기 어렵습니다.

그 결과 주문 조회가 아닌 질문에도 Tool을 호출하거나, 주문번호 대신 사용자 질문 전체를 `query`에 넣는 식의 잘못된 호출이 생기기 쉽습니다.

핵심은 모델이 약해서가 아니라, Tool 인터페이스가 너무 흐리다는 점입니다.


### 4.5 create_agent

Tool 정의 문법을 봤다면, 이제 그 Tool을 에이전트 실행에 연결할 차례입니다.  
`create_agent`는 모델이 사용자 요청을 읽고, 필요하면 Tool을 호출한 뒤, Tool 결과를 바탕으로 최종 답변을 만들게 해줍니다.

예를 들어 `주문번호 1001 배송 상태 알려줘`라는 요청은 모델 지식만으로 답하기 어렵습니다.  
이런 경우 주문 상태를 조회하는 Tool을 에이전트에 연결합니다.

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[lookup_order_status],
    system_prompt="주문 배송 상태 질문은 lookup_order_status 도구로 확인한 뒤 답하세요.",
)
```

핵심 흐름은 다음과 같습니다.

```text
사용자 질문
-> 모델이 Tool이 필요한지 판단
-> 필요하면 Tool 호출
-> Tool 결과를 바탕으로 최종 답변 생성
```

에이전트를 실행할 때는 보통 `messages`에 사용자 메시지를 넣어 전달합니다.

```python
order_agent.invoke({
    "messages": [
        {"role": "user", "content": "주문번호 1001 배송 상태 알려줘."}
    ]
})
```

`role`은 누가 보낸 메시지인지, `content`는 실제 요청 문장입니다.

#### 주요 파라미터

`create_agent()`는 최소한 `model`만 있으면 만들 수 있습니다. Tool을 쓰는 에이전트에서는 보통 `tools`와 `system_prompt`를 함께 넣습니다.

| 파라미터 | 필수 여부 | 무엇인가 | 역할 |
|----------|-----------|-----------|------|
| `model` | 필수 | 호출할 채팅 모델입니다. 문자열이나 `ChatOpenAI` 같은 모델 객체를 넣습니다. | 사용자의 요청을 읽고 Tool 사용 여부와 최종 답변을 판단합니다. |
| `tools` | 선택 | 에이전트가 호출할 수 있는 함수나 Tool 목록입니다. | 최신 정보 조회, DB 검색, 계산, 외부 시스템 실행처럼 모델 밖의 작업을 맡깁니다. 없으면 Tool 호출 없이 모델만 실행됩니다. |
| `system_prompt` | 선택 | 모델에게 먼저 주는 역할과 규칙입니다. 문자열이나 `SystemMessage`를 넣습니다. | 말투, 답변 기준, Tool 사용 조건을 정합니다. 예: 주문 상태는 반드시 `lookup_order_status`로 확인합니다. |
| `middleware` | 선택 | 에이전트 실행 중간에 끼워 넣는 처리입니다. | 모델 호출 전후, Tool 호출 전후에 로깅, 검증, 동적 프롬프트, Tool 제한 같은 공통 로직을 적용합니다. |
| `response_format` | 선택 | 최종 답변을 특정 구조로 받기 위한 스키마입니다. | Pydantic 모델 등을 넣으면 결과가 `structured_response`에 검증된 형태로 담깁니다. 고객 문의 분류, 정보 추출처럼 후처리가 필요한 경우에 씁니다. |
| `state_schema` | 선택 | 에이전트 상태에 추가로 저장할 값의 구조입니다. | 기본 상태는 `messages` 중심입니다. 여기에 `user_name`, `model_call_count`처럼 실행 중 누적할 값을 더할 때 사용합니다. |
| `context_schema` | 선택 | 실행할 때 함께 넘기는 런타임 정보의 구조입니다. | `user_id`, 권한, API 키, DB 연결 정보처럼 Tool이나 middleware가 읽어야 하지만 대화 기록으로 저장할 필요는 없는 값을 다룹니다. |
| `checkpointer` | 선택 | 에이전트 상태를 저장하고 다시 불러오는 저장 장치입니다. | 같은 `thread_id`의 대화를 이어가야 할 때 사용합니다. 로컬 실습에서는 `InMemorySaver`를 자주 씁니다. |

`state_schema`는 대화 중 바뀌며 저장되는 값, `context_schema`는 실행 시 바깥에서 주입하는 값이라고 구분하면 쉽습니다.


아래 셀은 주문 조회 Tool을 정의하고 `create_agent()`에 연결합니다. 마지막 줄에서는 최종 답변만 먼저 확인합니다.


In [ ]:
# 주문번호로 배송 상태를 조회하는 Tool을 에이전트에 연결합니다.
from langchain.agents import create_agent  # Tool을 연결한 에이전트를 만듭니다.
from langchain.tools import tool  # 함수를 Tool로 등록합니다.

@tool
def lookup_order_status(order_id: str) -> str:
    """주문번호로 배송 상태를 확인해야 할 때 사용합니다."""
    orders = {
        "1001": "배송 중입니다. 오늘 오후 도착 예정입니다.",
        "1002": "결제 확인 후 상품을 준비 중입니다.",
    }
    return orders.get(order_id, "해당 주문번호를 찾을 수 없습니다.")

model = ChatOpenAI(model="gpt-4.1-nano")  # 가볍게 실습하기 좋은 OpenAI 채팅 모델입니다.

order_agent = create_agent(
    model=model,  # 모델이 사용자 요청을 보고 Tool 사용 여부를 판단합니다.
    tools=[lookup_order_status],  # 이번 예제에서는 주문 조회 Tool 하나만 엽니다.
    system_prompt="주문 배송 상태 질문은 lookup_order_status 도구로 확인한 뒤 답하세요.",
)

response = order_agent.invoke(
    {"messages": [{"role": "user", "content": "주문번호 1001 배송 상태 알려줘."}]},
    lf_config,
)
print(response["messages"][-1].content)

에이전트 실행 결과는 문자열 하나가 아니라 `messages`가 들어 있는 딕셔너리입니다.  
최종 답변만 보면 중간에 Tool이 어떻게 호출됐는지 보이지 않으므로, 첫 실행 결과에서 바로 전체 메시지를 순서대로 확인합니다.


In [ ]:
# 전체 메시지를 보면 Tool 호출과 Tool 실행 결과가 어떤 순서로 쌓였는지 확인할 수 있습니다.
for index, message in enumerate(response["messages"], start=1):
    print(f"\n[{index}] {type(message).__name__}")
    print("content:", repr(message.content))

    if getattr(message, "tool_calls", None):
        print("tool_calls:", message.tool_calls)

    if getattr(message, "tool_call_id", None):
        print("tool_call_id:", message.tool_call_id)

이 실행에서는 보통 다음 순서로 메시지가 쌓입니다.

```text
HumanMessage: 사용자 질문
AIMessage: 모델이 Tool 호출을 결정함
ToolMessage: Tool 실행 결과
AIMessage: Tool 결과를 반영한 최종 답변
```

즉, `response["messages"][-1].content`는 사용자에게 보여줄 최종 답변이고, 중간 메시지를 보면 모델이 어떤 Tool을 어떤 입력값으로 호출했는지 확인할 수 있습니다.


### 4.6 파일을 읽는 Tool 실습

Tool은 API만 호출하는 기능이 아닙니다. 로컬 파일을 읽어 답변에 필요한 근거를 가져올 수도 있습니다.

아래 예제에서는 `data/` 폴더에 있는 두 종류의 자료를 사용합니다.

- `data/shop_policy.txt`: 쇼핑몰 배송, 교환, 환불, 쿠폰 규정
- `data/customer_inquiries.csv`: 실제 고객 문의가 쌓여 있는 CSV

에이전트는 질문을 보고 **규정을 확인해야 하면 정책 파일 Tool**, **고객 문의 데이터를 봐야 하면 CSV Tool**을 선택합니다.

실행 순서는 파일 경로 확인 -> Tool 정의 -> Tool 단독 테스트 -> 에이전트 연결 -> 질문 실행입니다.


먼저 사용할 파일 경로를 확인합니다. 경로가 잘못되면 뒤의 Tool도 파일을 읽지 못합니다.


In [ ]:
# data 폴더에 미리 준비된 파일 경로를 확인합니다.
from pathlib import Path
import csv

data_dir = Path("data")
policy_path = data_dir / "shop_policy.txt"
inquiry_path = data_dir / "customer_inquiries.csv"

print(policy_path.resolve())
print(inquiry_path.resolve())

파일 Tool을 만들 때 필요한 라이브러리를 불러옵니다.


In [ ]:
# 파일을 읽는 Tool과 에이전트를 준비합니다.
from langchain.agents import create_agent
from langchain.tools import tool

첫 번째 Tool은 쇼핑몰 규정 txt 파일을 읽어 옵니다.


In [ ]:
# 규정 질문에 사용할 Tool입니다.
@tool
def read_shop_policy(question: str) -> str:
    """쇼핑몰 배송, 교환, 환불, 쿠폰 같은 운영 규정을 확인해야 할 때 사용합니다."""
    return policy_path.read_text(encoding="utf-8")

두 번째 Tool은 고객 문의 CSV에서 질문과 관련 있는 행만 추려 문자열로 돌려줍니다.


In [ ]:
# 고객 문의 데이터 질문에 사용할 Tool입니다.
@tool
def read_customer_inquiries(category: str) -> str:
    """고객 문의 CSV에서 배송, 교환, 환불, 쿠폰 관련 실제 문의 사례를 찾아야 할 때 사용합니다."""
    with inquiry_path.open("r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    keywords = [word for word in category.lower().replace(",", " ").split() if len(word) >= 2]
    matched = [
        row for row in rows
        if any(word in row["category"].lower() or word in row["message"].lower() for word in keywords)
    ]
    if not matched:
        matched = rows

    return "\n".join(
        f"{row['id']} | {row['category']} | {row['customer']} | {row['message']}"
        for row in matched[:5]
    )

Tool은 에이전트에 연결하기 전에도 `.invoke()`로 직접 실행해볼 수 있습니다.  
아래 셀에서는 고객 문의 CSV Tool이 `배송` 관련 문의를 잘 찾는지 먼저 확인합니다.


In [ ]:
read_customer_inquiries.invoke('배송')

두 Tool이 준비되면 에이전트에 함께 등록합니다. 이제 모델은 질문을 보고 규정 파일을 읽을지 CSV를 볼지 선택할 수 있습니다.


In [ ]:
# 두 Tool을 모두 등록합니다.
shopping_agent = create_agent(
    model=model,
    tools=[read_shop_policy, read_customer_inquiries],
    system_prompt=(
        "당신은 쇼핑몰 고객지원 에이전트입니다. "
        "규정 확인이 필요하면 read_shop_policy를 사용하고, "
        "실제 고객 문의 목록을 확인해야 하면 read_customer_inquiries를 사용하세요. "
        "답변에는 어떤 자료를 근거로 판단했는지 짧게 포함하세요."
    ),
) 

첫 번째 질문은 쇼핑몰 규정 파일에서 확인해야 하는 내용입니다. 실행 결과의 `ToolMessage`에서 `read_shop_policy`가 선택됐는지 확인합니다.


In [ ]:
# 규정 질문은 read_shop_policy Tool을 사용해야 합니다.
policy_question = "착용 흔적이 있는 상품도 교환 가능한지 쇼핑몰 규정에서 확인해줘."

policy_result = shopping_agent.invoke(
    {"messages": [{"role": "user", "content": policy_question}]},
    lf_config,
)

print("질문:", policy_question)
print("최종 답변:")
print(policy_result["messages"][-1].content)

policy_result["messages"]


두 번째 질문은 고객 문의 CSV에서 실제 문의 사례를 찾아야 하는 내용입니다. 실행 결과의 `ToolMessage`에서 `read_customer_inquiries`가 선택됐는지 확인합니다.


In [ ]:
# 고객 문의 질문은 read_customer_inquiries Tool을 사용해야 합니다.
inquiry_question = "배송 지연 관련 고객 문의에는 어떤 내용이 있는지 CSV에서 찾아 요약해줘."

inquiry_result = shopping_agent.invoke(
    {"messages": [{"role": "user", "content": inquiry_question}]},
    lf_config,
)

print("질문:", inquiry_question)
print("최종 답변:")
print(inquiry_result["messages"][-1].content)

inquiry_result["messages"]

## 5. Structured Output

Structured Output은 Tool과 다른 문제를 해결합니다.  
Tool이 바깥 정보나 기능을 연결한다면, Structured Output은 최종 답을 사람이 읽는 문장 대신 **프로그램이 바로 쓸 수 있는 구조화 데이터**로 받는 문제를 다룹니다.

즉, Structured Output은 Tool이 없어도 사용할 수 있습니다.  
모델 결과를 다음 단계 코드나 시스템이 바로 사용해야 할 때 특히 유용합니다.



### 5.1 자유 텍스트 vs 구조화된 출력

<img src="image/자유텍스트와구조화된출력.png" width="700">


| 방식 | 예시 | 장점 | 단점 |
|------|------|------|------|
| 자유 텍스트 | "주문 ORD-1001 배송이 늦어져서 고객이 환불을 요청했습니다." | 사람이 읽기 자연스러움 | 후처리와 시스템 연동이 어려움 |
| 구조화된 출력 | `{"order_id": "ORD-1001", "request_type": "refund", "urgency": "high"}` | 자동화, 검증, 저장, 분기 처리에 유리 | 스키마 설계가 필요함 |




### 5.2 Pydantic

위 표에서 구조화된 출력은 정해진 칸을 가진 데이터라고 봤습니다.  
Pydantic은 그 칸의 이름, 타입, 허용 값을 Python 클래스로 적는 도구입니다.

먼저 고객 문의 결과가 어떤 필드를 가져야 하는지 Pydantic 클래스로 정해봅니다.  
값이 이 모양에 맞으면 객체가 만들어지고, 맞지 않으면 에러가 납니다.

- `Field(...)`: 각 칸이 무엇을 뜻하는지 설명합니다.
- `Literal[...]`: 허용 가능한 값을 제한합니다.
- `ValidationError`: 들어온 값이 규칙에 맞지 않을 때 발생하는 에러입니다.

```python
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

class CustomerInquiry(BaseModel):  # BaseModel을 상속하면 필드 타입과 허용 값을 검사할 수 있습니다.
    order_id: str = Field(description="주문번호")  # Field(...)로 각 필드 의미를 설명합니다.
    request_type: Literal["shipping", "exchange", "refund"] = Field(description="문의 유형")  # Literal[...]로 허용 값을 제한합니다.
    urgency: Literal["low", "medium", "high"] = Field(description="긴급도")
```

먼저 Pydantic 클래스를 단독으로 사용해보고, 그다음 같은 클래스를 `create_agent()`의 `response_format`에 연결합니다.




In [ ]:
# 먼저 Pydantic 클래스를 단독으로 사용해봅니다.
from typing import Literal  # 긴급도처럼 정해진 선택지를 표현합니다.
from pydantic import BaseModel, Field, ValidationError

class CustomerInquiry(BaseModel):  # BaseModel을 상속하면 아래 필드 규칙으로 값을 검사합니다.
    customer_name: str = Field(description="고객 이름")  # 누가 문의했는지 저장합니다.
    order_id: str = Field(description="주문번호")  # 어떤 주문에 대한 문의인지 저장합니다.
    request_type: Literal["shipping", "exchange", "refund"] = Field(description="문의 유형")  # 후속 분기에 바로 쓰기 좋습니다.
    urgency: Literal["low", "medium", "high"] = Field(description="긴급도")
    summary: str = Field(description="문의 내용 요약")

In [ ]:
# 성공 예시: 필드 이름과 타입이 맞으면 Pydantic 객체가 만들어집니다.
valid_inquiry = CustomerInquiry(
    customer_name="김민수",
    order_id="ORD-1001",
    request_type="refund",
    urgency="high",
    summary="배송 지연으로 환불 가능 여부를 문의함",
)

print(valid_inquiry)
print(valid_inquiry.model_dump())  # Pydantic 객체를 파이썬 dict로 바꿉니다.


In [ ]:
# 실패 예시: 타입이나 허용 값이 맞지 않으면 ValidationError가 납니다.
try:
    invalid_inquiry = CustomerInquiry(
        customer_name="김민수",
        order_id="ORD-1001",
        request_type="cancel",  # shipping/exchange/refund 중 하나여야 합니다.
        urgency="urgent",  # low/medium/high 중 하나여야 합니다.
        summary=["배송 지연", "환불 문의"],  # str이어야 합니다.
    )
except ValidationError as error:
    print("[검증 실패]")
    print(error)


Pydantic으로 데이터 모양과 검증 방식을 확인했습니다.  
이제 같은 `CustomerInquiry` 클래스를 에이전트의 `response_format`에 연결하면, 모델 답변도 이 구조에 맞춰 받을 수 있습니다.



In [ ]:
# Pydantic 클래스를 response_format에 연결합니다.
from langchain.agents import create_agent  # 구조화 출력을 반환할 에이전트를 만듭니다.

extraction_agent = create_agent(
    model=model,  # 자유 답변 대신 구조화된 객체를 만들 모델입니다.
    tools=[],  # 이번 문제는 Tool 없이 추출만 합니다.
    response_format=CustomerInquiry,  # 결과를 이 스키마에 맞춰 돌려받습니다.
    system_prompt="사용자 메시지에서 고객 문의 정보를 정확히 추출하세요.",
)

In [ ]:
# 사용자 문장에서 티켓 정보를 추출합니다.
ticket = extraction_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "안녕하세요, 김민수입니다. 주문 ORD-1001 배송이 아직 도착하지 않았고 환불 가능 여부를 확인하고 싶습니다.",
            }
        ]
    },
    lf_config,
)

structured_ticket = ticket["structured_response"]  # 구조화 결과는 별도 키에서 꺼냅니다.

In [ ]:
# 구조화 결과를 여러 형태로 확인합니다.
print("[Pydantic 객체]")
print(structured_ticket)
print("\n[dict 형태]")
print(structured_ticket.model_dump())  # 파이썬 dict로 바꿉니다.
print("\n[JSON 형태]")
print(structured_ticket.model_dump_json(indent=2))  # JSON 문자열로도 확인할 수 있습니다.


`response_format=CustomerInquiry`를 지정하면 모델의 최종 결과가 `CustomerInquiry` 객체로 반환됩니다.  
앞에서 직접 만든 Pydantic 객체처럼 `model_dump()`로 `dict`, `model_dump_json()`으로 JSON 문자열로 바꿀 수 있습니다.  
즉, Structured Output은 답변을 더 길고 예쁘게 쓰는 기능이 아니라, **다음 시스템이 바로 사용할 수 있는 결과 형식**을 만드는 기능입니다.


### 5.3 Structured Output 활용 예시

예:
- 문의를 `주제/긴급도/담당부서`로 분류하기
- 제안서를 `목표/기간/필수기술` 항목으로 추출하기
- 회의록을 `결정사항/담당자/마감일` 구조로 저장하기



#### 연습 문제: Structured Output 에이전트 만들기

고객 문의 문장을 읽고, 상담 시스템이 바로 저장할 수 있는 구조화 데이터로 바꾸는 에이전트를 만들어보세요.  
목표는 자유 텍스트 답변을 받는 것이 아니라, `structured_response`에서 Pydantic 객체를 꺼내는 것입니다.

상황:
- 쇼핑몰 고객지원팀은 문의 내용을 자동으로 분류해 티켓 시스템에 저장하려고 합니다.
- 사용자는 자연어로 문의를 남깁니다.
- 에이전트는 문의에서 필요한 항목을 추출해 정해진 구조로 반환해야 합니다.

요구사항:
- `SupportTicket` Pydantic 모델을 정의합니다.
- 필드는 `customer_name`, `order_id`, `category`, `priority`, `summary`로 구성합니다.
- `category`는 `shipping`, `exchange`, `refund`, `other` 중 하나로 제한합니다.
- `priority`는 `low`, `medium`, `high` 중 하나로 제한합니다.
- `create_agent()`를 사용해 에이전트를 만들고, `response_format=SupportTicket`을 연결합니다.
- `invoke()`에는 `{"messages": [{"role": "user", "content": ...}]}` 형태로 입력합니다.
- 실행 후 `result["structured_response"].model_dump()`로 결과를 확인합니다.

예시 입력:

```text
안녕하세요. 박지민입니다. 주문 ORD-2040으로 받은 상품 색상이 잘못 왔습니다. 교환을 빨리 진행하고 싶어요.
```




In [ ]:
# Structured Output 에이전트를 직접 완성해보세요.
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


class SupportTicket(BaseModel):
    # customer_name, order_id, category, priority, summary 필드를 정의해보세요.
    # category는 shipping/exchange/refund/other 중 하나로 제한합니다.
    # priority는 low/medium/high 중 하나로 제한합니다.
    pass


model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
system_prompt = None  # 문의에서 어떤 정보를 추출해야 하는지 적어보세요.

ticket_agent = None  # create_agent(..., response_format=SupportTicket, ...) 형태로 완성해보세요.

practice_input = {
    "messages": [
        {
            "role": "user",
            "content": "안녕하세요. 박지민입니다. 주문 ORD-2040으로 받은 상품 색상이 잘못 왔습니다. 교환을 빨리 진행하고 싶어요.",
        }
    ]
}

result = None  # ticket_agent.invoke(practice_input) 형태로 실행해보세요.
ticket = None  # result["structured_response"]에서 구조화 결과를 꺼내보세요.

print(ticket.model_dump())
print(ticket.model_dump_json(indent=2))


<details>
<summary>정답 보기</summary>

```python
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


class SupportTicket(BaseModel):
    customer_name: str = Field(description="고객 이름")
    order_id: str = Field(description="주문번호")
    category: Literal["shipping", "exchange", "refund", "other"] = Field(description="문의 유형")
    priority: Literal["low", "medium", "high"] = Field(description="처리 우선순위")
    summary: str = Field(description="문의 내용 요약")


model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

ticket_agent = create_agent(
    model=model,
    tools=[],
    response_format=SupportTicket,
    system_prompt=(
        "사용자 문의에서 고객명, 주문번호, 문의 유형, 우선순위, 요약을 추출하세요. "
        "상품이 잘못 왔거나 교환 처리가 필요한 경우 category는 exchange로 분류하세요."
    ),
)

result = ticket_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "안녕하세요. 박지민입니다. 주문 ORD-2040으로 받은 상품 색상이 잘못 왔습니다. 교환을 빨리 진행하고 싶어요.",
            }
        ]
    }
)

ticket = result["structured_response"]
print(ticket.model_dump())
print(ticket.model_dump_json(indent=2))
```

</details>




## 6. 에이전트 실행과 메모리

앞에서는 Tool을 붙인 에이전트와 Structured Output을 각각 봤습니다.  
이제 여러 Tool이 함께 쓰이는 실행 결과를 같은 방식으로 읽고, 같은 대화 흐름을 이어가는 메모리를 봅니다.  
마지막 실습에서는 Tool + Structured Output + Memory를 함께 붙인 통합 예제를 다룹니다.



### 6.1 여러 Tool을 가진 에이전트 예제

4.5에서는 주문 조회 Tool 하나를 가진 에이전트 실행 결과를 읽었습니다.  
이번에는 Tool이 두 개 이상 있을 때도 같은 방식으로 결과를 읽어봅니다.

에이전트는 보통 `{"messages": [...]}` 형태로 실행합니다.  
`invoke()`는 최종 결과를 한 번에 받고, `stream()`은 Tool 선택과 실행 과정을 중간 단계까지 보여줍니다.


In [ ]:
# Tool 두 개를 등록해 에이전트가 상황에 맞게 직접 선택하게 만듭니다.
from datetime import datetime  # 현재 시각을 만들 때 사용합니다.
from zoneinfo import ZoneInfo  # 서울 시간대를 지정할 때 사용합니다.
from langchain.agents import create_agent  # Tool을 쓸 수 있는 에이전트를 만듭니다.
from langchain.tools import tool  # 일반 함수를 에이전트가 쓸 수 있는 Tool로 등록합니다.

@tool
def get_korean_time() -> str:
    """현재 서울 시간을 반환합니다."""  # docstring이 Tool 설명으로 쓰입니다.
    now = datetime.now(ZoneInfo("Asia/Seoul"))  # 서울 기준 현재 시각을 구합니다.
    return now.strftime("%Y-%m-%d %H:%M:%S")  # 사람이 읽기 쉬운 문자열로 바꿉니다.

@tool
def add_numbers(a: int, b: int) -> int:
    """두 정수를 더합니다. 계산이 필요할 때 사용하세요."""
    return a + b  # 계산 결과를 그대로 돌려줍니다.

In [ ]:
# 두 Tool을 에이전트에 연결합니다.
basic_agent = create_agent(
    model=model,  # 어떤 모델이 Tool 사용 여부를 판단할지 정합니다.
    tools=[get_korean_time, add_numbers],  # 에이전트가 선택할 수 있는 Tool 목록입니다.
    system_prompt="당신은 쇼핑몰 운영 보조 에이전트입니다. 시간과 계산이 필요하면 반드시 도구를 사용하세요.",
)


아래 셀에서는 에이전트를 실행하고 반환된 `result` 전체를 먼저 확인합니다.


In [ ]:
# 한 질문 안에서 시간 Tool과 계산 Tool이 함께 쓰이는지 확인합니다.
result = basic_agent.invoke(
    {"messages": [{"role": "user", "content": "서울 기준 현재 시간과 쿠폰 3000원 + 적립금 2000원 합계를 알려줘."}]},
    lf_config,
)

result


### 6.2 여러 Tool 호출 결과 읽기

4.5에서 본 것처럼 에이전트 실행 결과의 핵심은 `messages`입니다.  
여기서는 한 질문 안에서 시간 조회 Tool과 계산 Tool이 함께 쓰일 때 메시지가 어떻게 쌓이는지 확인합니다.


2.2에서 본 LCEL 체인은 `StrOutputParser`까지 붙여 최종 문자열을 바로 받았습니다.  
반면 에이전트는 중간 실행 과정까지 확인할 수 있도록 메시지 목록을 함께 돌려줍니다.


Tool을 쓴 실행에서는 보통 `HumanMessage` -> `AIMessage`의 `tool_calls` -> `ToolMessage` -> 최종 `AIMessage` 순서로 메시지가 쌓입니다.


먼저 반환값의 키와 메시지 개수를 확인합니다.


In [ ]:
print(result.keys())
print("messages 개수:", len(result["messages"]))

첫 번째 메시지는 사용자가 보낸 `HumanMessage`입니다. 여기서는 원래 질문이 `content`에 들어 있습니다.


In [ ]:
result["messages"][0]

두 번째 메시지는 모델의 `AIMessage`입니다. Tool을 호출하기로 결정한 경우에는 아직 최종 답변을 만든 것이 아니어서 `content`가 비어 있을 수 있습니다. 이때는 `tool_calls`에 어떤 Tool을 어떤 인자로 부르려 했는지가 들어갑니다.


In [ ]:
result["messages"][1]

`tool_calls`만 따로 보면 모델이 선택한 Tool 이름과 입력값을 바로 확인할 수 있습니다.


In [ ]:
result["messages"][1].tool_calls

세 번째와 네 번째 메시지는 실제 Tool 실행 결과인 `ToolMessage`입니다. `name`에는 실행된 Tool 이름이, `content`에는 Tool이 반환한 값이 들어갑니다.


In [ ]:
result["messages"][2]

In [ ]:
result["messages"][3]

마지막 메시지는 Tool 결과를 바탕으로 모델이 다시 작성한 최종 답변입니다. 사용자 화면에는 보통 이 `content`만 보여줍니다.


In [ ]:
result["messages"][-1].content

### 6.3 스트리밍으로 에이전트 흐름 보기

방금 셀들에서는 실행이 끝난 뒤 `messages`에 쌓인 결과를 확인했습니다.  
`stream()`을 쓰면 실행이 끝나기를 기다리지 않고, 모델 판단과 Tool 실행 업데이트를 순서대로 받아볼 수 있습니다.



In [ ]:
# stream_mode="updates"는 실행 중 업데이트를 단계별로 보여줍니다.
for chunk in basic_agent.stream(
    {"messages": [{"role": "user", "content": "지금 시간만 알려줘."}]},  # 간단한 질문으로 흐름만 확인합니다.
    stream_mode="updates",  # 모델 판단과 Tool 실행 업데이트를 순서대로 받습니다.
    config=lf_config,):
    
    print(chunk)

### 6.4 메모리

앞에서는 한 번의 요청 안에서 Tool을 고르고 답을 만드는 흐름을 봤습니다.  
하지만 같은 에이전트를 여러 번 호출한다고 해서 이전 요청이 자동으로 이어지는 것은 아닙니다.

예를 들어 첫 번째 요청에서 `내 이름은 민수야`라고 말하고, 두 번째 요청에서 `내 이름이 뭐였지?`라고 물어도,  
두 번째 요청에 첫 번째 요청의 내용이 들어가지 않으면 모델은 그 이름을 알 수 없습니다.

대화가 이어진다는 것은 단순히 같은 에이전트를 다시 호출한다는 뜻이 아니라, 이전 대화 상태를 다음 요청의 Context에 다시 넣어준다는 뜻입니다.  
LangGraph 기반 에이전트에서는 보통 `checkpointer`가 상태를 저장하고, `thread_id`가 어느 대화 상태를 다시 불러올지 정합니다.

즉, `thread_id` 없이 매번 새 요청처럼 실행하면 에이전트는 이전에 했던 말을 기억하지 못합니다.


### 6.5 단기 메모리와 장기 메모리

메모리는 보통 유지 범위에 따라 나눠 생각합니다.

| 구분 | 설명 | 예시 |
|------|------|------|
| 단기 메모리 | 같은 대화 안에서 유지되는 정보 | 방금 말한 이름, 직전 질문, 현재 작업 흐름 |
| 장기 메모리 | 여러 대화에 걸쳐 남겨둘 정보 | 사용자 선호, 자주 쓰는 형식, 과거 패턴 |

초기 에이전트 실습에서는 먼저 **단기 메모리**만 이해해도 충분합니다.  
오래 남겨둘 정보와 현재 대화용 정보를 섞으면, 필요한 정보와 오래된 정보가 함께 들어와 판단이 흐려질 수 있습니다.


### 6.6 `thread_id`와 checkpointer

메모리를 쓸 때는 **어디에 저장할지**와 **어느 대화를 꺼낼지**를 함께 정해야 합니다.

| 개념 | 역할 | 확인할 점 |
|------|------|-----------|
| `checkpointer` | 대화 상태를 저장하고 다시 불러오는 저장 장치 | 여러 대화의 상태를 저장할 수 있습니다. |
| `thread_id` | checkpointer 안에서 특정 대화 흐름을 구분하는 키 | 같은 값을 쓰면 같은 대화가 이어지고, 다른 값을 쓰면 다른 대화로 분리됩니다. |

즉, `checkpointer`는 저장소이고 `thread_id`는 그 저장소 안에서 어느 대화 상태를 사용할지 고르는 식별자입니다.
같은 에이전트라도 `thread_id`가 다르면 서로 다른 대화로 취급합니다.

`InMemorySaver`는 LangGraph가 제공하는 체크포인터 구현이며, 이 예제에서는 대화 상태를 메모리에 저장하는 용도로 사용합니다.


In [ ]:
# checkpointer를 붙이면 같은 thread_id 안에서 대화 맥락을 이어갈 수 있습니다.
from langgraph.checkpoint.memory import InMemorySaver  # 메모리 안에 대화 상태를 저장합니다.

memory_agent = create_agent(
    model=model,  # 기억을 유지할 에이전트입니다.
    tools=[get_korean_time],  # 시간 Tool은 그대로 사용할 수 있게 둡니다.
    checkpointer=InMemorySaver(),  # 세션 상태를 메모리에 저장합니다.
    system_prompt="당신은 학습 코치입니다. 사용자가 알려준 자기소개를 기억하세요.",
)

config = {"configurable": {"thread_id": "student-1"}}  # 같은 thread_id면 같은 대화 흐름으로 이어집니다.
config.update(lf_config)

first = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 민수이고, 나는 개발자야."}]},  # 먼저 기억할 정보를 알려줍니다.
    config,
)
second = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "내 직업이 뭐였는지 기억해?"}]},  # 같은 thread_id로 후속 질문을 보냅니다.
    config,
)

print(second["messages"][-1].content)

같은 `thread_id`를 유지하면 checkpointer가 같은 대화 상태를 다시 불러옵니다.  
그래서 첫 번째 턴의 정보가 다음 턴에 이어집니다.

다른 `thread_id`를 쓰면 같은 checkpointer를 사용하더라도 별도의 대화 상태로 저장되고 복원됩니다.

아래 흐름만 기억하면 됩니다.

```text
현재 messages + thread_id
-> checkpointer에서 해당 thread_id의 상태를 복원
-> agent 실행
-> 새 messages/state를 같은 thread_id 아래에 저장
```


### 6.7 메모리는 컨텍스트로 다시 들어온다

메모리는 모델이 따로 꺼내 읽는 별도 공간이 아닙니다.  
checkpointer가 저장해 둔 이전 대화 상태를 불러오면, 그 내용이 다시 `messages`에 포함되어 모델에게 전달됩니다.

즉, 메모리는 **과거 대화를 다음 호출의 컨텍스트로 다시 넣어주는 공급원**입니다.

같은 `thread_id`를 쓰면 이전 대화가 컨텍스트에 포함되고, 다른 `thread_id`를 쓰면 이전 대화가 포함되지 않습니다.


<img src="image/memory_context_flow.svg" width="760">




In [ ]:
# thread_id를 바꾸면 이전 대화가 컨텍스트에 들어오지 않습니다.
other_config = {"configurable": {"thread_id": "student-2"}}  # 다른 사용자나 다른 세션이라고 생각하면 됩니다.
other_config.update(lf_config)

isolated = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "내 직업이 뭐였는지 기억해?"}]},  # 앞선 소개 없이 바로 질문합니다.
    other_config,
)

print(isolated["messages"][-1].content)


## 7. 1일차 미니 실습



### 7.1 실습 문제 1

먼저 방금 본 Structured Output 개념을 직접 손으로 정의해봅니다.  
이 실습은 **Structured Output 스키마 설계와 추출 프롬프트 작성**을 연습하기 위한 문제입니다.

사용자의 요구를 구조화된 JSON 형태로 추출하는 에이전트를 설계해보세요.

필드별 타입 힌트:
- `project_name`: `str`
- `priority`: `Literal["low", "medium", "high"]`
- `duration_weeks`: `int`
- `must_have_features`: `list[str]`

코드 셀의 `pass`와 `None`을 채운 뒤 실행해보세요.

예시 입력:
- 물류창고 출고 검수 자동화 프로젝트를 6주 안에 시범 구축하고 싶어.
- 우선순위는 높고, 바코드 스캔과 이상 탐지, 대시보드는 꼭 들어가야 해.

In [ ]:
# 필요한 타입 힌트 4개와 프롬프트를 직접 채워보세요.
from typing import Literal
from pydantic import BaseModel, Field  # Field(description=...)로 필드 의미를 적습니다.
from langchain.agents import create_agent  # response_format에 스키마를 연결합니다.
from langchain_openai import ChatOpenAI  # 실습 셀 단독 실행을 위해 모델을 여기서 만듭니다.

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)  # 이 실습에서 사용할 기본 채팅 모델입니다.

class AutomationProjectRequest(BaseModel):
    # 프로젝트 이름(`project_name`, 문자열), 우선순위(`priority`, low/medium/high),
    # 예상 기간(`duration_weeks`, 정수), 필수 기능 목록(`must_have_features`, 문자열 리스트)을 직접 작성해보세요.
    # 각 필드에는 타입 힌트와 Field(description=...)를 함께 적습니다.
    pass


In [ ]:
# 스키마와 프롬프트를 에이전트에 연결합니다.
extract_prompt = None  # 사용자 요구를 위 스키마 형태로 추출하라는 프롬프트를 직접 적어보세요.

practice_agent = create_agent(
    model=model,  # 이 셀 안에서 바로 만든 모델을 연결합니다.
    tools=[],  # 이번 문제는 Tool 없이 구조화 추출만 합니다.
    response_format=AutomationProjectRequest,  # 위에서 만든 Pydantic 스키마를 연결합니다.
    system_prompt=extract_prompt,
)


In [ ]:
# 추출할 사용자 요청을 준비합니다.
practice_input = {
    "messages": [
        {
            "role": "user",
            "content": "쇼핑몰 고객 문의 자동 분류 프로젝트를 4주 안에 시범 구축하고 싶어. 우선순위는 높고, 주문 조회와 환불 규정 안내, 상담 대시보드는 꼭 들어가야 해.",
        }
    ]
}

practice_config = {}  # 추가 실행 설정이 없으면 빈 dict를 넘깁니다.


In [ ]:
# 위 셀의 pass와 None을 채운 뒤 실행하세요.
practice_result = practice_agent.invoke(practice_input, practice_config)  # 구조화 결과는 structured_response에서 확인합니다.
print(practice_result["structured_response"].model_dump())  # dict 형태로 봅니다.
print(practice_result["structured_response"].model_dump_json(indent=2))  # JSON 문자열로도 확인합니다.

<details>
<summary>정답 보기</summary>

```python
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5-nano", temperature=0)

class AutomationProjectRequest(BaseModel):
    project_name: str = Field(description="프로젝트 이름")
    priority: Literal["low", "medium", "high"] = Field(description="우선순위")
    duration_weeks: int = Field(description="예상 기간(주 단위)")
    must_have_features: list[str] = Field(description="반드시 포함할 기능")

extract_prompt = "사용자 요구를 쇼핑몰 고객지원 자동화 프로젝트 구조 데이터로 추출하세요."

practice_agent = create_agent(
    model=model,
    tools=[],
    response_format=AutomationProjectRequest,
    system_prompt=extract_prompt,
)

practice_input = {
    "messages": [
        {
            "role": "user",
            "content": "쇼핑몰 고객 문의 자동 분류 프로젝트를 4주 안에 시범 구축하고 싶어. 우선순위는 높고, 주문 조회와 환불 규정 안내, 상담 대시보드는 꼭 들어가야 해.",
        }
    ]
}

practice_config = {}
result = practice_agent.invoke(practice_input, practice_config)

print(result["structured_response"].model_dump())
print(result["structured_response"].model_dump_json(indent=2))
```

</details>




### 7.2 실습 문제 2

이번에는 Tool 설명 품질과 입력 스키마를 **구체적인 고객지원 시나리오**로 연습해봅니다.  
쇼핑몰 고객지원 챗봇이 문의 문장을 읽고 `배송`, `교환`, `환불` 중 하나로 분류하는 Tool을 직접 만들어보세요.

과제 요구사항:
- 입력 스키마 이름은 `CustomerRequestInput`으로 둡니다.
- 필드는 `request: str` 하나만 받습니다.
- Tool 이름은 `classify_customer_request`로 둡니다.
- docstring에는 이 Tool을 언제 써야 하는지 적습니다.
- 반환값은 `배송`, `교환`, `환불` 중 하나만 나오게 만듭니다.

예시 요청:
- `송장 번호가 아직 안 나왔어요` -> `배송`
- `사이즈가 맞지 않아 교환하고 싶어요` -> `교환`
- `반품 접수 후 환불은 언제 되나요` -> `환불`




In [ ]:
# 쇼핑몰 고객지원 챗봇용 분류 Tool을 직접 완성해보세요.
from pydantic import BaseModel, Field  # 입력 스키마를 좁게 정의합니다.
from langchain.tools import tool  # 일반 함수를 Tool로 등록합니다.

class CustomerRequestInput(BaseModel):
    request: str = Field(description=None)  # 예: "분류할 고객 문의 문장"

@tool(args_schema=CustomerRequestInput)
def classify_customer_request(request: str) -> str:
    """이 Tool을 언제 써야 하는지 한 문장으로 적어보세요."""
    # 아래 규칙을 참고해 조건문을 완성해보세요.
    # - "배송", "송장", "도착"이 들어가면 "배송"
    # - "교환" 또는 "사이즈"가 들어가면 "교환"
    # - 그 밖에는 "환불"
    return None

<details>
<summary>정답 보기</summary>

```python
from pydantic import BaseModel, Field
from langchain.tools import tool

class CustomerRequestInput(BaseModel):
    request: str = Field(description="분류할 고객 문의")

@tool(args_schema=CustomerRequestInput)
def classify_customer_request(request: str) -> str:
    """고객 문의 내용을 읽고 배송, 교환, 환불 중 하나로 분류해야 할 때 사용합니다."""
    if "배송" in request or "송장" in request or "도착" in request:
        return "배송"
    if "교환" in request or "사이즈" in request:
        return "교환"
    return "환불"
```

핵심 포인트:
- Tool 이름보다 `docstring`이 모델의 선택 기준을 더 분명하게 만들어줍니다.
- 입력 필드를 좁게 정의하면 모델이 Tool 사용 시 덜 헷갈립니다.
- 반환값은 다음 단계에서 바로 쓰기 쉽게 단순하게 두는 편이 좋습니다.

</details>




### 7.3 실습 문제 3: 통합 예제

마지막으로 앞에서 본 개념을 하나로 묶어봅니다.  
이 실습은 **Tool + Structured Output + Memory + `thread_id` 관계**를 한 번에 체감하기 위한 통합 예제입니다.

`Tool`, `Structured Output`, `Memory`를 함께 쓰는 기초 에이전트를 직접 완성해보세요.

시나리오:
- 고객 문의 문장을 구조화합니다.
- 필요하면 주문 처리 가이드를 조회하는 Tool을 사용합니다.
- 같은 `thread_id`로 후속 요청까지 이어서 처리합니다.

`SupportRequest` 필드는 아래 타입으로 정의합니다.

| 필드 | 타입 | 가이드 |
|------|------|------|
| `order_id` | `str` | 주문번호는 `1001`, `ORD-1001`처럼 숫자만 있거나 문자/하이픈이 섞일 수 있으므로 문자열로 둡니다. |
| `request_summary` | `str` | 고객 문의를 한두 문장으로 요약합니다. |
| `urgency` | `Literal["low", "medium", "high"]` | `낮음`, `보통`, `높음` 같은 표현을 세 값 중 하나로 바꿉니다. |
| `action_plan` | `str` | Tool 결과와 사용자 요청을 반영한 처리 계획을 적습니다. |
| `needs_human_review` | `bool` | 담당자 검토가 필요하면 `True`, 아니면 `False`입니다. |

각 필드는 `Field(description=...)`로 의미를 적어주세요.

코드 셀의 `None`과 빈칸을 채운 뒤 실행해보세요.




In [ ]:
# Tool + Structured Output + Memory를 결합한 에이전트를 직접 완성해보세요.
from typing import Literal
from pydantic import BaseModel, Field  # 구조화 출력 스키마를 만듭니다.
from langchain.agents import create_agent  # Tool, Memory, Structured Output을 한 번에 묶습니다.
from langchain.tools import tool  # 주문 처리 안내 함수를 Tool로 등록합니다.
from langgraph.checkpoint.memory import InMemorySaver  # 같은 thread_id의 대화를 이어가기 위해 사용합니다.
from langchain_openai import ChatOpenAI  # 실습 셀 단독 실행을 위해 모델을 여기서 만듭니다.

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)  # 응답과 Tool 사용 여부를 판단할 기본 모델입니다.

class SupportRequest(BaseModel):
    # order_id: str, request_summary: str, urgency: Literal["low", "medium", "high"]
    # action_plan: str, needs_human_review: bool 필드를 추가하세요.
    pass


In [ ]:
# 주문별 처리 가이드를 조회하는 Tool입니다.
@tool
def lookup_order_guide(order_id: str) -> str:
    """이 Tool을 언제 써야 하는지 한 문장으로 직접 적어보세요."""
    guides = {
        "ORD-1001": "ORD-1001은 배송 지연 상태입니다. 예상 도착일을 확인하고 지연 안내와 보상 쿠폰 여부를 검토합니다.",
        "ORD-1002": "ORD-1002는 사이즈 교환 문의입니다. 재고 확인 후 교환 가능 여부와 회수 일정을 안내합니다.",
        "ORD-1003": "ORD-1003은 환불 검수 대기 상태입니다. 검수 완료 후 결제 수단으로 환불됩니다.",
    }
    return guides.get(order_id, None)  # 찾지 못했을 때 안내 문구를 넣고 싶다면 직접 바꿔보세요.


In [ ]:
# Tool, 구조화 출력, 메모리를 한 에이전트에 연결합니다.
integrated_agent = create_agent(
    model=model,  # 이 셀 안에서 만든 모델을 연결합니다.
    tools=[lookup_order_guide],  # 주문 처리 가이드 조회 Tool을 등록합니다.
    response_format=SupportRequest,  # 결과를 구조화된 객체로 받습니다.
    checkpointer=InMemorySaver(),  # 같은 thread_id의 맥락을 이어가기 위해 필요합니다.
    system_prompt=None,  # 구조화 추출 기준과 Tool 사용 조건을 직접 적어보세요.
)


In [ ]:
# 같은 thread_id로 첫 요청과 후속 요청을 이어서 실행합니다.
practice_config = {"configurable": {"thread_id": "support-practice-1"}}  # 같은 값을 유지해야 후속 요청의 "그 주문"을 이어서 이해합니다.

first_message = {
    "messages": [
        {
            "role": "user",
            "content": "주문 ORD-1001 배송이 아직 도착하지 않았어. 오늘 안으로 처리 요청을 만들고 싶어. 긴급도는 높음으로 보고, 필요하면 고객지원팀 검토로 넘겨줘.",
        }
    ]
}

followup_message = {
    "messages": [
        {
            "role": "user",
            "content": "그 주문은 오늘 오후 안에 고객에게 안내해야 해. 조치 요약에도 그 내용을 반영해줘.",
        }
    ]
}

In [ ]:
# 위 셀의 None과 빈칸을 채운 뒤 실행해보세요.
first_result = integrated_agent.invoke(first_message, practice_config)  # 첫 번째 요청을 구조화합니다.
print("[첫 번째 요청]")
print(first_result["structured_response"].model_dump())

followup_result = integrated_agent.invoke(followup_message, practice_config)  # 같은 thread_id로 후속 요청을 이어서 보냅니다.
print("\n[후속 요청]")
print(followup_result["structured_response"].model_dump())

<details>
<summary>정답 보기</summary>

```python
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

class SupportRequest(BaseModel):
    order_id: str = Field(description="문의 대상 주문번호")
    request_summary: str = Field(description="사용자가 말한 문의 내용 요약")
    urgency: Literal["low", "medium", "high"] = Field(description="처리 긴급도")
    action_plan: str = Field(description="주문 처리 가이드와 사용자 요청을 반영한 조치 요약")
    needs_human_review: bool = Field(description="고객지원팀이나 담당자 검토가 필요한지 여부")

@tool
def lookup_order_guide(order_id: str) -> str:
    """주문번호를 기준으로 처리 가이드가 필요할 때 사용합니다."""
    guides = {
        "ORD-1001": "ORD-1001은 배송 지연 상태입니다. 예상 도착일을 확인하고 지연 안내와 보상 쿠폰 여부를 검토합니다.",
        "ORD-1002": "ORD-1002는 사이즈 교환 문의입니다. 재고 확인 후 교환 가능 여부와 회수 일정을 안내합니다.",
        "ORD-1003": "ORD-1003은 환불 검수 대기 상태입니다. 검수 완료 후 결제 수단으로 환불됩니다.",
    }
    return guides.get(order_id, "해당 주문 가이드를 찾을 수 없습니다.")

integrated_agent = create_agent(
    model=model,
    tools=[lookup_order_guide],
    response_format=SupportRequest,
    checkpointer=InMemorySaver(),
    system_prompt=(
        "사용자의 고객 문의를 SupportRequest 형식으로 정리하세요. "
        "주문별 처리 가이드가 필요하면 lookup_order_guide 도구를 사용하세요. "
        "긴급도 표현은 low, medium, high 중 하나로 변환하세요. "
        "사용자가 고객지원팀 검토를 요청하거나 긴급도가 높으면 needs_human_review를 true로 설정하세요. "
        "후속 요청에서 '그 주문'처럼 이전 대상을 가리키면 같은 thread_id의 이전 대화 내용을 이어서 사용하세요."
    ),
)

practice_config = {"configurable": {"thread_id": "support-practice-1"}}

first_result = integrated_agent.invoke(first_message, practice_config)
print("[첫 번째 요청]")
print(first_result["structured_response"].model_dump())

followup_result = integrated_agent.invoke(followup_message, practice_config)
print("[후속 요청]")
print(followup_result["structured_response"].model_dump())
```

확인할 점:
- `lookup_order_guide`의 docstring이 Tool 사용 기준이 됩니다.
- `response_format=SupportRequest`로 최종 결과가 구조화됩니다.
- 같은 `thread_id`를 사용하므로 후속 요청의 `그 주문`이 앞선 `ORD-1001` 맥락과 이어집니다.

</details>




## 8. 1일차 정리

| 구분 | 핵심 내용 |
|------|-------------------|
| OpenAI API | ChatGPT 웹 구독과 별도로 쓰는 사용량 기반 호출 방식 |
| 프롬프트 구조 | `System / User / Assistant`처럼 역할을 나눠 보낸다 |
| LangChain | 메시지 구성, 모델 호출, 출력 정리를 코드 흐름으로 묶는 도구 |
| Context | 모델이 이번 호출에서 참고하는 정보 전체 |
| Tool | 최신 정보 조회, 계산, 실행 같은 바깥 연결이 필요할 때 쓴다 |
| `create_agent` | 모델이 필요하면 Tool을 고르고, 결과를 바탕으로 다시 답하는 실행 구조를 만든다 |
| system_prompt | 답변 스타일뿐 아니라 Tool 사용 기준도 정할 수 있다 |
| Structured Output | 최종 답을 프로그램이 바로 쓰는 구조화 데이터로 받는다 |
| Memory | `thread_id`와 `checkpointer`로 상태를 이어간다 |

```text
Tool -> 모델만으로 안 되는 문제를 해결
create_agent -> Tool 사용을 포함한 에이전트 실행 구조 생성
system_prompt -> 필요하면 특정 작업에서 Tool 사용을 더 강하게 유도
Structured Output -> 최종 결과를 구조화 데이터로 반환
통합 예제 -> Tool + Structured Output + Memory를 함께 적용
```


